In [16]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from libpysal.weights import full2W
from esda.moran import Moran_Local
from PIL import Image
import os
from pyprojroot import here

In [17]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [18]:
crime = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', encoding='latin-1', index_col=0, dtype=str)
gdf = gpd.read_file(RAW_DATA_DIR/"geographic/municipalities/00mun.shp")

In [19]:
crime["CVEGEO"] = crime["CVEGEO"].astype(str).str.zfill(5)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)
gdf["CVEGEO"] = gdf["CVEGEO"].astype(str).str.zfill(5)

common_ids = sorted(set(adj.index) & set(crime["CVEGEO"]) & set(gdf["CVEGEO"]))
print(f"Common municipalities across all 3 sources: {len(common_ids)}")

adj = adj.loc[common_ids, common_ids]
gdf = gdf.set_index("CVEGEO").loc[common_ids].reset_index()

Common municipalities across all 3 sources: 2478


In [20]:
w = full2W(adj.values.astype(int), ids=common_ids)
w.transform = "r"
print(f"True islands: {w.islands}")

QUADRANT_LABELS = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
ALPHA = 0.05
PERMUTATIONS = 999

CLUSTER_COLORS = {
    "HH": "#d7191c",   # red    - high surrounded by high
    "LL": "#2c7bb6",   # blue   - low surrounded by low
    "HL": "#fdae61",   # orange - high surrounded by low (outlier)
    "LH": "#abd9e9",   # light blue - low surrounded by high (outlier)
    "ns": "#f0f0f0",   # grey   - not significant
    "island": "#ffffff",
}

years = sorted(crime["year"].unique())

True islands: []


In [21]:
frame_paths = []
all_years_results = []

for year in years:
    yr_data = crime[crime["year"] == year].set_index("CVEGEO").reindex(common_ids)
    y = yr_data["crime_count"].values.astype(float)

    lisa = Moran_Local(y, w, permutations=PERMUTATIONS, seed=42)
    sig = lisa.p_sim < ALPHA
    cluster = np.where(sig, [QUADRANT_LABELS[q] for q in lisa.q], "ns")
    cluster = np.where(np.isnan(lisa.z_sim), "island", cluster)  # mark true islands

    year_df = pd.DataFrame({
        "CVEGEO": common_ids,
        "year": year,
        "cluster": cluster,
    })
    all_years_results.append(year_df)

    # merge onto geometry
    plot_gdf = gdf.merge(year_df, on="CVEGEO", how="left")

    # ---- plot ----
    fig, ax = plt.subplots(figsize=(10, 10))
    for cluster_type, color in CLUSTER_COLORS.items():
        subset = plot_gdf[plot_gdf["cluster"] == cluster_type]
        subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)

    plot_gdf.boundary.plot(ax=ax, linewidth=0.1, color="#999999")

    ax.set_title(f"Crime Rate LISA Clusters — {year}", fontsize=16, fontweight="bold")
    ax.axis("off")

    legend_patches = [
        mpatches.Patch(color=CLUSTER_COLORS["HH"], label="High-High"),
        mpatches.Patch(color=CLUSTER_COLORS["LL"], label="Low-Low"),
        mpatches.Patch(color=CLUSTER_COLORS["HL"], label="High-Low (outlier)"),
        mpatches.Patch(color=CLUSTER_COLORS["LH"], label="Low-High (outlier)"),
        mpatches.Patch(color=CLUSTER_COLORS["ns"], label="Not significant"),
    ]
    ax.legend(handles=legend_patches, loc="lower left", fontsize=10, frameon=True)

    plt.savefig(RESULTS_DATA_DIR/f"esda/LISA_{year}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    frame_paths.append(RESULTS_DATA_DIR/f"esda/LISA_{year}.png")
    print(f"Saved frame for {year}")

lisa_results = pd.concat(all_years_results, ignore_index=True)

C:\Users\franc\AppData\Local\Temp\ipykernel_91528\2056489919.py:27: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2015


C:\Users\franc\AppData\Local\Temp\ipykernel_91528\2056489919.py:27: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2016


C:\Users\franc\AppData\Local\Temp\ipykernel_91528\2056489919.py:27: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


...

Saved frame for 2023


C:\Users\franc\AppData\Local\Temp\ipykernel_91528\2056489919.py:27: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2024


C:\Users\franc\AppData\Local\Temp\ipykernel_91528\2056489919.py:27: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2025


In [22]:
frames = [Image.open(p) for p in frame_paths]
# pad all frames to same size (bbox_inches='tight' can vary slightly)
max_w = max(f.width for f in frames)
max_h = max(f.height for f in frames)
padded = []
for f in frames:
    bg = Image.new("RGB", (max_w, max_h), "white")
    bg.paste(f, ((max_w - f.width) // 2, (max_h - f.height) // 2))
    padded.append(bg)

gif_path = RESULTS_DATA_DIR/"esda/yearly_lisa_clusters.gif"
os.makedirs(RESULTS_DATA_DIR/"esda", exist_ok=True)
padded[0].save(
    gif_path,
    save_all=True,
    append_images=padded[1:],
    duration=1000,   # ms per frame
    loop=0,
)
print(f"GIF saved to {gif_path}")

GIF saved to C:\Users\franc\OneDrive\Documents\Repositories\organized-crime-prediction\results\esda\yearly_lisa_clusters.gif
